In [1]:
import os 
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))
sys.path.append(os.path.dirname(os.getcwd()))

import time
import random
from pprint import pprint
from glob import glob
import numpy as np
import pickle
import matplotlib.pyplot as plt
import cv2
from matplotlib.patches import FancyArrowPatch

import pyrender

import pygarment as pyg
import trimesh
import PIL
from PIL import Image

from analysis_utils import visualize_meshes_plotly, v_id_map, plot_panel_info

DATASET_ROOT_PATH = "/home/hjp/VTO2025/GarmentCodeData"
GARMENT_ROOT_PATH = os.path.join(DATASET_ROOT_PATH, "GarmentCodeData_v2")
BODY_ROOT_PATH = os.path.join(DATASET_ROOT_PATH, "body_mesh")
MEAN_ALL_BODY_PATH = os.path.join(DATASET_ROOT_PATH, "neutral_body/mean_all.obj")

default_body_mesh = trimesh.load(MEAN_ALL_BODY_PATH)

print("body vertices", default_body_mesh.vertices.shape)
print("body faces", default_body_mesh.faces.shape)

# BODY_TYPE = "random_body"
BODY_TYPE = "default_body"

def fff(garment_path) :
    garment_id = os.path.basename(garment_path)
    if os.path.exists(
        os.path.join(garment_path, f"{garment_id}_fltrd_vis_seam_line_dict.pkl")
    ) and os.path.exists(
        os.path.join(garment_path, f"rendered_front.png")
    ):
        return True
    else :
        return False

garment_path_list = list(filter(
    fff,
    sorted(list(filter(
        os.path.isdir,
        glob(os.path.join(GARMENT_ROOT_PATH, "*", BODY_TYPE, "*"))
    )))
))

body vertices (23751, 3)
body faces (47500, 3)


In [2]:
problematic_data_path_list = []
for garment_path in garment_path_list :
    try :    
        garment_id = os.path.basename(garment_path)
        rendered_image_dict = {}
        for side in ["front", "back", "left", "right"] :
            rendered_image_dict[side] = Image.open(os.path.join(garment_path, f"rendered_{side}.png"))
        with open(os.path.join(garment_path, f"{garment_id}_fltrd_vis_seam_line_dict.pkl"), "rb") as f :
            fltrd_vis_seam_line_dict = pickle.load(f)

        with open(os.path.join(garment_path, f"{garment_id}_projected_vertex_pose.pkl"), "rb") as f :
            projected_vertex_pose_dict = pickle.load(f)

        with open(os.path.join(garment_path, f"{garment_id}_vertex_visibility_mask.pkl"), "rb") as f :
            vertex_visibility_mask_dict = pickle.load(f)

        side_list = random.sample(
            ["front", "back", "left", "right"],
            random.randint(1, 4)
        )
    except Exception as e :
        print(e)
        problematic_data_path_list.append(garment_path)

print(len(problematic_data_path_list))


0


In [7]:
for path in problematic_data_path_list :
    garment_id = os.path.basename(path)
    
    rendered_image_dict = {}
    for side in ["front", "back", "left", "right"] :
        try :
            img_path = os.path.join(path, f"rendered_{side}.png")
            os.remove(img_path)
        except Exception as e :
            print(e)
    
    fltrd_vis_seam_line_dict_path = os.path.join(path, f"{garment_id}_fltrd_vis_seam_line_dict.pkl")
    os.remove(fltrd_vis_seam_line_dict_path)

    projected_vertex_pose_dict_path = os.path.join(path, f"{garment_id}_projected_vertex_pose.pkl")
    os.remove(projected_vertex_pose_dict_path)

    vertex_visibility_mask_dict_path = os.path.join(path, f"{garment_id}_vertex_visibility_mask.pkl")
    os.remove(vertex_visibility_mask_dict_path)


[Errno 2] No such file or directory: '/home/hjp/VTO2025/GarmentCodeData/GarmentCodeData_v2/garments_5000_0/default_body/rand_XLH6TSUX84/rendered_front.png'
[Errno 2] No such file or directory: '/home/hjp/VTO2025/GarmentCodeData/GarmentCodeData_v2/garments_5000_0/default_body/rand_XLH6TSUX84/rendered_back.png'
[Errno 2] No such file or directory: '/home/hjp/VTO2025/GarmentCodeData/GarmentCodeData_v2/garments_5000_0/default_body/rand_XLH6TSUX84/rendered_left.png'
[Errno 2] No such file or directory: '/home/hjp/VTO2025/GarmentCodeData/GarmentCodeData_v2/garments_5000_0/default_body/rand_XLH6TSUX84/rendered_right.png'


In [5]:
os.listdir(
    '/home/hjp/VTO2025/GarmentCodeData/GarmentCodeData_v2/garments_5000_0/default_body/rand_XLH6TSUX84/'
)

['rand_XLH6TSUX84_body_measurements.yaml',
 'depth_right.npy',
 'rand_XLH6TSUX84_projected_vertex_pose.pkl',
 'rand_XLH6TSUX84_pattern.png',
 'rand_XLH6TSUX84_specification.json',
 'rand_XLH6TSUX84_design_params.yaml',
 'rand_XLH6TSUX84_sim_segmentation.txt',
 'rand_XLH6TSUX84_vertex_visibility_mask.pkl',
 'rand_XLH6TSUX84_render_back.png',
 'rand_XLH6TSUX84_texture.png',
 'rand_XLH6TSUX84_orig_lens.pickle',
 'rand_XLH6TSUX84_boxmesh.ply',
 'depth_left.npy',
 'rand_XLH6TSUX84_vertex_labels.yaml',
 'depth_back.npy',
 'rand_XLH6TSUX84_sim.ply',
 'rand_XLH6TSUX84_render_front.png',
 'rand_XLH6TSUX84_pattern.svg',
 'depth_front.npy',
 'rand_XLH6TSUX84_fltrd_vis_seam_line_dict.pkl']